# Lecture 7.3: Defining Custom Agents with `AgentDefinition`

The Bike Repair Co-op is a cooperative workshop run by volunteers. Members bring in bikes, volunteers fix them, and the shop keeps its records in a few plain files: work orders, a parts inventory and a volunteer roster.

So far a main agent could only hand work to a general-purpose helper. In this notebook you build **your own specialists**, each with its own job description, instructions, tools and model:

| Specialist | Job | Tools | Defined |
|---|---|---|---|
| `parts-clerk` | Reports stock levels from the inventory | `Read`, `Glob`, `Grep` (read-only) | In code |
| `order-writer` | Drafts purchase orders as text files | `Read`, `Write` | In code |
| `shift-planner` | Matches work orders to volunteers | `Read`, `Glob`, `Grep` | In a markdown file |

### What the cells do

| Cells | Purpose |
|---|---|
| 1 to 3 | Setup: install the SDK, load your API key, choose the model |
| 4 to 5 | Create the co-op files and compute the correct answer with plain Python |
| 6 to 7 | Define `parts-clerk` and meet the camelCase rule |
| 8 to 10 | Automatic invocation versus explicit invocation |
| 11 to 13 | A second specialist, and proof on disk that tool restrictions hold |
| 14 to 15 | An agent defined in a markdown file, loaded from the filesystem |

### The mental model to keep in mind

`query()` is not a function that returns a value. It returns an **async stream of messages** while the agent loop runs on its own. Tool calls and tool results arrive as separate messages, and the final answer arrives last on the `ResultMessage`. A delegation to a subagent is just another tool call in that stream, and messages produced inside a subagent carry a `parent_tool_use_id`.

Agents are not deterministic. Wording, message counts and even the choice to delegate can differ between runs. Each cell tells you what to look for, so focus on patterns rather than exact text.

## Cell 1: Install the Claude Agent SDK

This cell installs the `claude-agent-sdk` Python package, the only package this notebook needs.

| Part | Meaning |
|---|---|
| `!pip install` | The leading `!` tells Colab to run the line in the shell instead of as Python |
| `claude-agent-sdk==0.2.174` | The package, pinned to one exact version |
| `-q` | Quiet mode, so only errors are printed |

**Why pin the version?** Pinning keeps every example reproducible. The SDK ships with the Claude Code command line tool bundled inside it, so a new release can change agent behavior. To use the latest release, run `pip install -U claude-agent-sdk` instead, or substitute any version you prefer. Newer versions may behave differently from the videos.

If the package is already installed in your current Colab session, pip notices and finishes almost immediately.

In [ ]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.174 -q

## Cell 2: Load your API key from Colab Secrets

The SDK reads your Anthropic API key from the `ANTHROPIC_API_KEY` environment variable. This cell copies the key from Colab's secret store into that variable, so the key never appears in the notebook itself.

**Add the secret in Colab (one time):**

1. Click the **key icon** in the left sidebar of Colab.
2. Click **Add new secret**.
3. Set the name to exactly `ANTHROPIC_API_KEY`.
4. Paste your API key as the value.
5. Switch **Notebook access** on for this notebook.

You create the key in the Claude Console at [platform.claude.com](https://platform.claude.com/) under **API Keys**.

**Running locally instead of Colab?** Skip this cell and set the environment variable in your terminal before starting Jupyter, for example `export ANTHROPIC_API_KEY=your-key-here`.

In [ ]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

This cell stores the model name in one variable, `MODEL_NAME`. **Changing this one line changes the model for the entire notebook.**

It matters more than usual here, because the variable is used in two kinds of places:

- every `ClaudeAgentOptions(model=MODEL_NAME, ...)`, which sets the **main agent's** model, and
- every `AgentDefinition(model=MODEL_NAME, ...)` and the agent file in Cell 14, which set each **specialist's** model.

This notebook uses Claude Haiku 5.5, which is fast and lightweight. Model names change with new releases, so the comment links to the models page with the current list.

In [ ]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-5-5"

## Cell 4: Create the Bike Repair Co-op files

This cell builds the workshop folder that every agent in this notebook works on. Everything is created in code from a fixed random seed, so every learner gets identical files.

```
/content/bikecoop/
    work_orders/WO-101.txt ... WO-140.txt    (40 work orders)
    inventory.csv                            (15 parts and their stock levels)
    volunteers.txt                           (6 volunteers, their skills and shift days)
```

**`inventory.csv`** is the file that matters most in this notebook. It has three columns:

| Column | Meaning |
|---|---|
| `part` | The part name |
| `in_stock` | How many the co-op has right now |
| `reorder_level` | The stock level at which the co-op should order more |

**Exactly four parts have `in_stock` below `reorder_level`.** That is the ground truth for the `parts-clerk` questions. One part sits exactly **at** its reorder level, which is not below it. That boundary case checks whether an agent reads the numbers carefully.

**The work orders and volunteers** are used at the end of the notebook, when the `shift-planner` agent matches a work order to a volunteer.

**A clean start.** The cell deletes and recreates the `bikecoop` folder. It also removes the `shift-planner.md` agent file that Cell 14 creates, if it exists from an earlier run. Without that step, a rerun of the earlier cells would quietly see an extra agent, as Cell 15 explains.

In [ ]:
import os
import random
import shutil

BASE = "/content/bikecoop"
WO_DIR = f"{BASE}/work_orders"
AGENT_FILE = "/content/.claude/agents/shift-planner.md"

# Start from a clean folder so every run produces identical files.
shutil.rmtree(BASE, ignore_errors=True)
os.makedirs(WO_DIR)

# Remove the file-defined agent from any earlier run (Cell 14 recreates it).
if os.path.exists(AGENT_FILE):
    os.remove(AGENT_FILE)

rng = random.Random(7)

bikes = [
    "Blue hybrid city bike", "Red steel road bike", "Green mountain bike",
    "Black folding bike", "Yellow cargo bike", "Silver commuter bike",
    "White step-through bike", "Orange kids bike, 20 inch wheels",
    "Grey gravel bike", "Purple tandem", "Teal touring bike",
    "Cream vintage three-speed", "Black BMX bike", "Maroon e-bike",
    "Lime green trail bike", "Pink kids bike, 16 inch wheels",
    "Brown utility bike", "Navy racing bike", "Olive hardtail mountain bike",
    "Gold single-speed bike",
]

owners = [
    "Anita Rao", "Marcus Bell", "Priya Nair", "Tom Okafor", "Lena Fischer",
    "Dev Malhotra", "Sofia Alvarez", "Kwame Mensah", "Yuki Tanaka",
    "Ravi Iyer", "Hannah Cole", "Omar Haddad", "Grace Liu", "Nikhil Shah",
    "Elena Petrova", "Samuel Wright", "Meera Pillai", "Jonas Berg",
    "Fatima Khan", "Arjun Menon",
]

volunteers = [
    ("Asha", "wheels and brakes", "Monday and Thursday"),
    ("Ben", "gears and drivetrain", "Tuesday and Saturday"),
    ("Chitra", "frames and welding", "Wednesday and Saturday"),
    ("Dinesh", "e-bike electrics", "Friday"),
    ("Esha", "general servicing", "Monday and Wednesday"),
    ("Farid", "kids bikes and fitting", "Thursday and Sunday"),
]

urgent_safety_problems = [
    "Front brake lever pulls to the handlebar with almost no stopping power",
    "Cracked weld near the fork crown",
    "Rear brake cable frayed and close to snapping",
    "Chain keeps jumping under load and has locked up once",
    "Front wheel wobbles because a bearing cone has come loose",
    "Handlebar stem clamp slips while riding",
    "Crank arm is loose on the spindle",
    "Rear wheel has a broken spoke and is badly out of true",
]

urgent_no_safety_problems = [
    "Owner wants a full tune-up before a charity ride on Saturday",
    "Paint scratched and rust starting at the dropouts",
    "Bell missing and basket bracket bent",
    "Saddle torn and owner starts a new commute tomorrow",
    "Kickstand broken so the bike keeps falling over",
]

high_safety_problems = [
    "Brake pads worn down to the wear line",
    "Rear light will not hold a charge",
    "Tyre sidewall bulging slightly",
    "Gear cable sticky and skipping gears",
    "Pedal bearing feels gritty",
    "Rear reflector missing",
]

routine_problems = [
    "Squeaky bottom bracket", "Flat tyre needs a new tube",
    "Gear indicator out of alignment", "Mudguard rattles over bumps",
    "Grips worn smooth", "Chain needs cleaning and lubrication",
    "Saddle height will not stay put", "Water bottle cage is loose",
    "Seasonal service requested", "Small wobble, wheel needs truing",
    "Brakes squeal in wet weather", "Handlebar tape unwinding",
]

history_notes = [
    "Intake check done and frame inspected for visible damage.",
    "Owner phoned to confirm the best day for collection.",
    "Spare parts shelf checked for matching sizes.",
    "Workshop stand booked for the next free slot.",
    "Photo of the fault added to the job folder.",
    "Test ride completed in the yard, no further faults noted.",
    "Cleaned the drivetrain before inspection started.",
    "Tyre pressure and quick-release levers checked.",
    "Owner asked for a quote before any parts are ordered.",
    "Handover checklist printed and clipped to the job card.",
    "Bike tagged and moved to the repair bay.",
    "Noted that the owner collects on weekday evenings.",
]

# Decide which order gets which role, using the seeded generator.
ids = list(range(101, 141))
rng.shuffle(ids)
urgent_safety_ids = ids[:5]
urgent_no_ids = ids[5:10]
high_safety_ids = ids[10:16]
routine_ids = ids[16:]

problem_for = {}
priority_for = {}
safety_for = {}

for number, problem in zip(urgent_safety_ids, rng.sample(urgent_safety_problems, 5)):
    problem_for[number], priority_for[number], safety_for[number] = problem, "urgent", "yes"

for number, problem in zip(urgent_no_ids, urgent_no_safety_problems):
    problem_for[number], priority_for[number], safety_for[number] = problem, "urgent", "no"

for number, problem in zip(high_safety_ids, high_safety_problems):
    problem_for[number], priority_for[number], safety_for[number] = problem, "high", "yes"

for number in routine_ids:
    problem_for[number] = rng.choice(routine_problems)
    priority_for[number] = rng.choice(["high", "normal", "normal", "low"])
    safety_for[number] = "no"

for number in range(101, 141):
    work_order_id = f"WO-{number}"
    volunteer_name = rng.choice(volunteers)[0]
    status = rng.choice(["open", "in progress", "waiting for parts"])

    text = (
        f"Work order: {work_order_id}\n"
        f"Bike: {rng.choice(bikes)}\n"
        f"Owner: {rng.choice(owners)}\n"
        f"Problem: {problem_for[number]}\n"
        f"Priority: {priority_for[number]}\n"
        f"Safety issue: {safety_for[number]}\n"
        f"Status: {status}\n"
        f"Assigned volunteer: {volunteer_name}\n"
        f"\n"
        f"Repair history log:\n"
    )

    # Add history lines until the file reaches its target size.
    target_length = rng.randint(1250, 1450)
    while True:
        entry = (
            f"2026-{rng.randint(1, 9):02d}-{rng.randint(1, 28):02d} | "
            f"{rng.choice(volunteers)[0]} | {rng.choice(history_notes)}\n"
        )
        if len(text) + len(entry) > target_length:
            break
        text += entry

    with open(f"{WO_DIR}/{work_order_id}.txt", "w", encoding="utf-8") as file:
        file.write(text)

# Parts inventory: (part, in_stock, reorder_level).
# Exactly four parts are below their reorder level.
# "gear cables" sits exactly at its reorder level, which is not below it.
parts = [
    ("brake pads", 14, 10), ("brake cables", 6, 8), ("inner tubes 700c", 22, 12),
    ("inner tubes 26 inch", 9, 12), ("chains", 7, 6), ("chain rings", 3, 4),
    ("spokes assorted", 120, 60), ("bar tape", 11, 6), ("saddles", 4, 3),
    ("pedals pairs", 7, 5), ("headset bearings", 8, 6), ("gear cables", 8, 8),
    ("rear lights", 12, 8), ("kickstands", 2, 4), ("grips pairs", 9, 6),
]

with open(f"{BASE}/inventory.csv", "w", encoding="utf-8") as file:
    file.write("part,in_stock,reorder_level\n")
    for part, in_stock, reorder_level in parts:
        file.write(f"{part},{in_stock},{reorder_level}\n")

with open(f"{BASE}/volunteers.txt", "w", encoding="utf-8") as file:
    for name, skills, shifts in volunteers:
        file.write(f"{name} | skills: {skills} | shifts: {shifts}\n")

print("Bike Repair Co-op files created.")

Bike Repair Co-op files created.


## Cell 5: Verify the files and compute the ground truth

Before any agent runs, this cell proves the files exist and works out the correct answer **with plain Python**, with no AI involved.

It does four things:

1. Prints the folder tree, with the 40 work orders summarized on one line.
2. Prints `inventory.csv` exactly as it is on disk.
3. Reads the CSV with Python's `csv` module and collects every part whose `in_stock` is below its `reorder_level`. The result is stored in `LOW_STOCK`.
4. Prints `volunteers.txt` and work order `WO-103`, which the `shift-planner` agent works with at the end of the notebook.

`LOW_STOCK` is **filesystem ground truth**. Later cells check each agent's answer against it, instead of judging the answer by how confident it sounds. The cell ends with an `assert` that exactly four parts are low. If it fails, rerun Cell 4.

In [ ]:
import csv

# 1. Folder tree
print("bikecoop/")
for name in sorted(os.listdir(BASE)):
    path = f"{BASE}/{name}"
    if os.path.isdir(path):
        files = sorted(os.listdir(path))
        print(f"    {name}/  ({len(files)} files, {files[0]} ... {files[-1]})")
    else:
        print(f"    {name}")

# 2. The inventory, exactly as it is on disk
print("\n--- inventory.csv ---")
with open(f"{BASE}/inventory.csv", encoding="utf-8") as file:
    print(file.read())

# 3. Ground truth: parts below their reorder level, computed with plain Python
LOW_STOCK = []
with open(f"{BASE}/inventory.csv", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        if int(row["in_stock"]) < int(row["reorder_level"]):
            LOW_STOCK.append(row["part"])

print("--- Ground truth: parts below their reorder level ---")
for part in LOW_STOCK:
    print(f"  {part}")

assert len(LOW_STOCK) == 4, "Expected exactly four low-stock parts. Rerun Cell 4."

# 4. Files the shift-planner agent will use later
print("\n--- volunteers.txt ---")
with open(f"{BASE}/volunteers.txt", encoding="utf-8") as file:
    print(file.read())

print("--- WO-103 (first 8 lines) ---")
with open(f"{WO_DIR}/WO-103.txt", encoding="utf-8") as file:
    print("".join(file.readlines()[:8]))

bikecoop/
    inventory.csv
    volunteers.txt
    work_orders/  (40 files, WO-101.txt ... WO-140.txt)

--- inventory.csv ---
part,in_stock,reorder_level
brake pads,14,10
brake cables,6,8
inner tubes 700c,22,12
inner tubes 26 inch,9,12
chains,7,6
chain rings,3,4
spokes assorted,120,60
bar tape,11,6
saddles,4,3
pedals pairs,7,5
headset bearings,8,6
gear cables,8,8
rear lights,12,8
kickstands,2,4
grips pairs,9,6

--- Ground truth: parts below their reorder level ---
  brake cables
  inner tubes 26 inch
  chain rings
  kickstands

--- volunteers.txt ---
Asha | skills: wheels and brakes | shifts: Monday and Thursday
Ben | skills: gears and drivetrain | shifts: Tuesday and Saturday
Chitra | skills: frames and welding | shifts: Wednesday and Saturday
Dinesh | skills: e-bike electrics | shifts: Friday
Esha | skills: general servicing | shifts: Monday and Wednesday
Farid | skills: kids bikes and fitting | shifts: Thursday and Sunday

--- WO-103 (first 8 lines) ---
Work order: WO-103
Bike: Brow

## Cell 6: Define the `parts-clerk` agent

A custom subagent is described by an `AgentDefinition`. It is a plain Python dataclass: building one does not start anything or call any model. It only describes a specialist that a main agent may later delegate to.

| Field | Required | What it does here |
|---|---|---|
| `description` | Yes | Tells the **main agent** when to use this specialist. The main agent reads it when it decides whether to delegate, so it should say what the agent does and when to use it. |
| `prompt` | Yes | The specialist's **own system prompt**. Only the subagent sees it. |
| `tools` | No | The only tools the specialist gets. If you leave it out, the agent inherits every tool available to subagents. |
| `model` | No | The model this specialist runs on. If you leave it out, the main agent's model is used. Here it is `MODEL_NAME`, so every agent in the notebook stays on the same model. |

**Design decisions:**

- **A short, specific description.** It names the job (stock levels) and the trigger (any question about parts or stock). Vague descriptions make routing guesswork.
- **A two or three sentence prompt.** The clerk is told it is read-only and must report only facts from `inventory.csv`.
- **Read-only tools.** `Read`, `Glob` and `Grep` let the clerk look at files but never change them. A tool that is not in this list does not exist for this agent at all. Cell 13 proves that on disk.

The cell prints the object so you can see every field, including the optional ones you did not set, which show as `None`.

In [ ]:
from claude_agent_sdk import AgentDefinition

parts_clerk = AgentDefinition(
    description=(
        "Checks the co-op's parts inventory and reports stock levels. "
        "Use for any question about parts or stock."
    ),
    prompt=(
        "You are the parts clerk at the Bike Repair Co-op. "
        "You are read-only: you never create or change files. "
        "Answer only with facts you read from /content/bikecoop/inventory.csv."
    ),
    tools=["Read", "Glob", "Grep"],
    model=MODEL_NAME,
)

print(parts_clerk)

AgentDefinition(description="Checks the co-op's parts inventory and reports stock levels. Use for any question about parts or stock.", prompt='You are the parts clerk at the Bike Repair Co-op. You are read-only: you never create or change files. Answer only with facts you read from /content/bikecoop/inventory.csv.', tools=['Read', 'Glob', 'Grep'], disallowedTools=None, model='claude-haiku-5-5', skills=None, memory=None, mcpServers=None, initialPrompt=None, maxTurns=None, background=None, effort=None, permissionMode=None)


## Cell 7: The camelCase trap

Python code normally uses snake_case names such as `disallowed_tools`. **`AgentDefinition` does not.** Its multi-word fields keep the camelCase spelling of the wire format the SDK sends to Claude Code:

`disallowedTools`, `mcpServers`, `initialPrompt`, `maxTurns`, `permissionMode`

`ClaudeAgentOptions`, the object that configures the main agent, does use snake_case. So the same idea is spelled differently depending on which object you are building.

| Concept | `AgentDefinition` (one specialist) | `ClaudeAgentOptions` (main agent) |
|---|---|---|
| Tools to remove | `disallowedTools` | `disallowed_tools` |
| Permission mode | `permissionMode` | `permission_mode` |

Because `AgentDefinition` is a dataclass, a misspelled keyword is not silently ignored. Python raises a `TypeError` the moment you construct it.

This cell does three things:

1. Builds an `AgentDefinition` with the snake_case keyword `disallowed_tools=` inside a `try` block, and prints the error.
2. Builds the same agent with the correct `disallowedTools=` and prints the field.
3. Prints the two-row comparison table above.

`disallowedTools` is a denylist: the agent inherits its tools as usual, minus the ones you list. The agent built here is a throwaway example and is not used again.

In [ ]:
# 1. The snake_case spelling fails at construction time.
try:
    AgentDefinition(
        description="Example agent.",
        prompt="Example prompt.",
        disallowed_tools=["Write"],
    )
except TypeError as error:
    print("TypeError:", error)

# 2. The camelCase spelling works.
no_write_agent = AgentDefinition(
    description="Example agent.",
    prompt="Example prompt.",
    disallowedTools=["Write"],
    model=MODEL_NAME,
)
print("\nCorrect spelling, disallowedTools =", no_write_agent.disallowedTools)

# 3. Same idea, two spellings.
print(f"\n{'Concept':<20}{'AgentDefinition':<20}{'ClaudeAgentOptions':<20}")
print("-" * 60)
print(f"{'Tools to remove':<20}{'disallowedTools':<20}{'disallowed_tools':<20}")
print(f"{'Permission mode':<20}{'permissionMode':<20}{'permission_mode':<20}")

TypeError: AgentDefinition.__init__() got an unexpected keyword argument 'disallowed_tools'. Did you mean 'disallowedTools'?

Correct spelling, disallowedTools = ['Write']

Concept             AgentDefinition     ClaudeAgentOptions  
------------------------------------------------------------
Tools to remove     disallowedTools     disallowed_tools    
Permission mode     permissionMode      permission_mode     


## Cell 8: Automatic invocation

Now the main agent gets a team member. Two options make that happen:

| Option | What it does |
|---|---|
| `agents={"parts-clerk": parts_clerk}` | Registers the specialist. The dictionary **key** is the agent's name, which is what Claude uses to refer to it. |
| `"Agent"` in `allowed_tools` | Claude delegates through a tool named `Agent`. Listing it auto-approves delegation, so no permission prompt is needed. |
| `env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"}` | Makes the main agent **wait** for each subagent to finish before it answers. |

The other options are the usual ones: `model=MODEL_NAME` for the main agent and `cwd="/content"` as its working directory.

**Why the `env` line matters.** By default, a subagent starts in the **background**. The main agent hands off the work and carries on, and the subagent's result reaches it later, as a notification in a following turn. A single `query()` call has no following turn, so without this line the run can end with an answer like "the parts-clerk is still working on it". The `env` option passes environment variables to Claude Code, and this variable turns background tasks off. Every cell in this notebook that delegates uses it.

**The prompt does not mention the agent.**

```
Which parts in /content/bikecoop/inventory.csv are below their reorder level?
```

That is deliberate. With no name in the prompt, the main agent has to decide for itself whether the question matches the clerk's `description`. This is called **automatic invocation**, and it is a real test of how good your description is.

Every message is collected in the list `auto_messages`, so Cell 9 can inspect the stream after the run. The final answer is printed from the `ResultMessage`.

**What to look for:** an answer listing low-stock parts. Whether the main agent delegated is not visible here. Cell 9 checks that. The main agent may decide the question is simple enough to answer by reading the file itself. That is not a bug. It is what automatic invocation means: the decision belongs to the model.

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

options = ClaudeAgentOptions(
    agents={"parts-clerk": parts_clerk},
    allowed_tools=["Read", "Glob", "Grep", "Agent"],
    env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},
    model=MODEL_NAME,
    cwd="/content",
)

auto_messages = []
async for message in query(
    prompt="Which parts in /content/bikecoop/inventory.csv are below their reorder level?",
    options=options,
):
    auto_messages.append(message)
    if isinstance(message, ResultMessage):
        print(message.result)

Four parts are below their reorder level:

| Part | In stock | Reorder level | Short by |
|---|---|---|---|
| brake cables | 6 | 8 | 2 |
| inner tubes 26 inch | 9 | 12 | 3 |
| chain rings | 3 | 4 | 1 |
| kickstands | 2 | 4 | 2 |

Gear cables are at exactly their reorder level (8 of 8), so they are not below it.


## Cell 9: Did the main agent delegate?

This cell defines three small helpers and uses them on Cell 8's messages. The later cells reuse them.

| Helper | What it returns |
|---|---|
| `agent_calls(messages)` | Every delegation in the stream. A delegation is a `ToolUseBlock` inside an `AssistantMessage` whose `name` is `"Agent"`. The helper also matches `"Task"`, the tool's older name, which still appears in some places. |
| `final_text(messages)` | The `result` text of the `ResultMessage`, or an empty string. |
| `check_low_stock(text)` | Which of the four ground-truth parts appear in a piece of text. |

For each delegation, the cell prints `block.input.get("subagent_type")`, the name of the agent the main agent chose. If there were no delegations, it prints `no delegation`.

Then it checks the final answer against `LOW_STOCK` from Cell 5. The check matches part names as they are written in the CSV. If the agent rewords a name (for example "26-inch tubes"), the check can miss it, so read the answer yourself as well.

**What to look for:**

- **`subagent_type: parts-clerk`** means the description did its job, and the main agent routed the question to the specialist.
- **`no delegation`** means the main agent answered by itself. Automatic invocation is a judgement call, and **the description is the only lever you have** over it. Cell 10 shows the reliable alternative.

Either way, the answer should name the four low-stock parts and should not list `gear cables` as low, because it is exactly at its reorder level. A good answer may still mention `gear cables` to explain why it does not count, so the cell prints a note when the name appears and leaves the judgement to you.

In [ ]:
from claude_agent_sdk import AssistantMessage, ToolUseBlock


def agent_calls(messages):
    """Return every Agent tool call in the stream (the tool was once named Task)."""
    calls = []
    for message in messages:
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, ToolUseBlock) and block.name in ("Agent", "Task"):
                    calls.append(block)
    return calls


def final_text(messages):
    """Return the result text of the run, or an empty string."""
    for message in messages:
        if isinstance(message, ResultMessage):
            return message.result or ""
    return ""


def check_low_stock(text):
    """Report which ground-truth parts appear in the text."""
    lowered = text.lower()
    found = [part for part in LOW_STOCK if part in lowered]
    print(f"Ground-truth parts found: {len(found)} of {len(LOW_STOCK)}")
    missing = [part for part in LOW_STOCK if part not in found]
    if missing:
        print(f"Missing: {missing}")
    if "gear cables" in lowered:
        print(
            "Note: the answer mentions 'gear cables', which is AT its reorder level. "
            "Read the answer to check it is not listed as low."
        )


print("--- Delegations in Cell 8 ---")
calls = agent_calls(auto_messages)
if calls:
    for call in calls:
        print(f"subagent_type: {call.input.get('subagent_type')}")
else:
    print("no delegation")

print("\n--- Checking the answer against the ground truth ---")
check_low_stock(final_text(auto_messages))

--- Delegations in Cell 8 ---
no delegation

--- Checking the answer against the ground truth ---
Ground-truth parts found: 4 of 4
Note: the answer mentions 'gear cables', which is AT its reorder level. Read the answer to check it is not listed as low.


## Cell 10: Explicit invocation

Same question, one change: the prompt now **names the agent**.

```
Use the parts-clerk agent to find which parts in /content/bikecoop/inventory.csv
are below their reorder level.
```

Naming the agent is called **explicit invocation**. Instead of asking the main agent to match the task to a description, you tell it which specialist to use. Everything else is unchanged: the same `parts_clerk` definition, the same tools, the same `env` setting so the main agent waits for the clerk, and the same model and working directory.

| | Automatic (Cell 8) | Explicit (this cell) |
|---|---|---|
| Prompt names the agent | No | Yes |
| Who picks the specialist | The main agent, by matching the `description` | You |
| Reliability | A judgement call | Much more dependable |

The cell prints the `subagent_type` of every delegation, the final answer, and the ground-truth check.

**What to look for:** `subagent_type: parts-clerk`. Compare it with Cell 9. If Cell 8 did not delegate, this is the clearest contrast in the notebook: naming the agent removes the guesswork.

In [ ]:
explicit_options = ClaudeAgentOptions(
    agents={"parts-clerk": parts_clerk},
    allowed_tools=["Read", "Glob", "Grep", "Agent"],
    env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},
    model=MODEL_NAME,
    cwd="/content",
)

explicit_messages = []
async for message in query(
    prompt=(
        "Use the parts-clerk agent to find which parts in "
        "/content/bikecoop/inventory.csv are below their reorder level."
    ),
    options=explicit_options,
):
    explicit_messages.append(message)

print("--- Delegations ---")
calls = agent_calls(explicit_messages)
if calls:
    for call in calls:
        print(f"subagent_type: {call.input.get('subagent_type')}")
else:
    print("no delegation")

print("\n--- Final answer ---")
answer = final_text(explicit_messages)
print(answer or "(no answer returned)")

print()
check_low_stock(answer)

--- Delegations ---
subagent_type: parts-clerk

--- Final answer ---
Four parts are below their reorder level in `/content/bikecoop/inventory.csv`:

| Part | In stock | Reorder level |
|---|---|---|
| brake cables | 6 | 8 |
| inner tubes 26 inch | 9 | 12 |
| chain rings | 3 | 4 |
| kickstands | 2 | 4 |

Gear cables are at exactly their reorder level (8 of 8), so they don't count as below it. The file has no separate ID column, so parts are identified by name.

Ground-truth parts found: 4 of 4
Note: the answer mentions 'gear cables', which is AT its reorder level. Read the answer to check it is not listed as low.


## Cell 11: Define the `order-writer` agent

The co-op now needs a second specialist that **can** write files. `order-writer` drafts purchase orders and saves them as text files.

| Field | Value | Why |
|---|---|---|
| `description` | Drafts purchase orders and saves them under `/content/bikecoop/orders/` | Tells the main agent this is the one to use when an order must be written |
| `prompt` | A short instruction to write clear purchase orders, one line per part | The specialist's own system prompt |
| `tools` | `["Read", "Write"]` | It needs `Read` to look at the inventory and `Write` to save the order |
| `model` | `MODEL_NAME` | Same model as every other agent |

Compare the two specialists side by side:

| Agent | Has `Write`? |
|---|---|
| `parts-clerk` | **No.** Its tools are `Read`, `Glob`, `Grep`. |
| `order-writer` | **Yes.** Its tools are `Read`, `Write`. |

That difference is what Cells 12 and 13 test.

The cell also creates the `orders` folder and empties it, so the existence checks in Cell 13 start from a known state. It then prints the folder contents, which should be empty.

In [ ]:
ORDERS_DIR = f"{BASE}/orders"

order_writer = AgentDefinition(
    description=(
        "Drafts purchase orders for parts the co-op needs and saves them as "
        "text files in /content/bikecoop/orders/. Use when an order must be written."
    ),
    prompt=(
        "You are the order writer at the Bike Repair Co-op. "
        "Write clear purchase orders with one line per part and the quantity to order. "
        "Save each order as a plain text file in /content/bikecoop/orders/."
    ),
    tools=["Read", "Write"],
    model=MODEL_NAME,
)

# Start with an empty orders folder so Cell 13 checks a known state.
shutil.rmtree(ORDERS_DIR, ignore_errors=True)
os.makedirs(ORDERS_DIR)

print(order_writer)
print(f"\nContents of {ORDERS_DIR}: {os.listdir(ORDERS_DIR)}")

AgentDefinition(description='Drafts purchase orders for parts the co-op needs and saves them as text files in /content/bikecoop/orders/. Use when an order must be written.', prompt='You are the order writer at the Bike Repair Co-op. Write clear purchase orders with one line per part and the quantity to order. Save each order as a plain text file in /content/bikecoop/orders/.', tools=['Read', 'Write'], disallowedTools=None, model='claude-haiku-5-5', skills=None, memory=None, mcpServers=None, initialPrompt=None, maxTurns=None, background=None, effort=None, permissionMode=None)

Contents of /content/bikecoop/orders: []


## Cell 12: Ask both specialists to write a file

One `query()` registers both specialists and gives each one a writing job:

- **`parts-clerk`** is asked to save the low-stock list to `orders/reorder_list.txt`. It has no `Write` tool.
- **`order-writer`** is asked to save a purchase order to `orders/purchase_order.txt`. It has `Write`.

**Why `"Write"` is in `allowed_tools`.** `allowed_tools` auto-approves tools for the run, including tool calls made inside subagents. With `Write` approved, a missing file cannot be blamed on a permission prompt. If `reorder_list.txt` does not appear, the only explanation left is the clerk's own `tools` list. That is what makes the result attributable.

**Why the `env` line matters even more here.** Without it, the run could end while `order-writer` is still working in the background. `purchase_order.txt` would then be missing for the wrong reason, and Cell 13 could not tell a restriction apart from a run that simply stopped early.

**Why the prompt ends with "Do not create any files yourself."** The main agent also has `Write` in this run. Without that sentence, it could quietly write `reorder_list.txt` on the clerk's behalf, and the demo would prove nothing. Cell 13 checks whether it obeyed.

The cell prints one line per tool call, labelled with who made it, and then the final answer. Messages from inside a subagent carry a `parent_tool_use_id`. Cell 13 uses it to work out exactly which subagent each call came from.

**What to look for:** a `Write` call labelled as coming from inside a subagent, and no `Write` call from the parts clerk. Read the final answer too. The main agent may report that the clerk could not save its file. Real agents vary, so focus on the pattern.

In [ ]:
from claude_agent_sdk import UserMessage

team_options = ClaudeAgentOptions(
    agents={"parts-clerk": parts_clerk, "order-writer": order_writer},
    allowed_tools=["Read", "Glob", "Grep", "Write", "Agent"],
    env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},
    model=MODEL_NAME,
    cwd="/content",
)

TEAM_PROMPT = (
    "Use the parts-clerk agent to check /content/bikecoop/inventory.csv and save "
    "the parts below their reorder level to /content/bikecoop/orders/reorder_list.txt. "
    "Then use the order-writer agent to draft a purchase order for those parts and "
    "save it to /content/bikecoop/orders/purchase_order.txt. "
    "Do not create any files yourself. Only the named agents may create files."
)

team_messages = []
try:
    async for message in query(prompt=TEAM_PROMPT, options=team_options):
        team_messages.append(message)
        if isinstance(message, AssistantMessage):
            who = "main agent" if message.parent_tool_use_id is None else "inside a subagent"
            for block in message.content:
                if isinstance(block, ToolUseBlock):
                    print(f"{who:<18} -> {block.name}")
except Exception as error:
    print(f"Run ended with an error: {error}")

print("\n--- Final answer ---")
print(final_text(team_messages) or "(no answer returned)")

main agent         -> Agent
inside a subagent  -> Read
main agent         -> Agent
inside a subagent  -> Read
inside a subagent  -> Write

--- Final answer ---
The purchase order is saved, but `reorder_list.txt` was not. The parts-clerk agent only has Read, Glob, and Grep, so it cannot write files.

**Low-stock parts** (from `/content/bikecoop/inventory.csv`):

| Part | In stock | Reorder level |
|---|---|---|
| Brake cables | 6 | 8 |
| Inner tubes 26 inch | 9 | 12 |
| Chain rings | 3 | 4 |
| Kickstands | 2 | 4 |

Gear cables are at their reorder level (8 of 8), so they are not included.

**Purchase order:** saved to `/content/bikecoop/orders/purchase_order.txt` by the order-writer agent. It orders the top-up amount for each part: 2 brake cables, 3 inner tubes, 1 chain ring, and 2 kickstands, 8 units in total. The inventory file has no supplier or cost columns, so supplier, unit prices, and totals are marked TBD.

**Still missing:** `/content/bikecoop/orders/reorder_list.txt`. I did no

## Cell 13: Ground truth on disk

Now check what really happened, in two steps.

**Step 1: filesystem ground truth.** The cell checks whether each file exists:

| File | Requested from | Expected |
|---|---|---|
| `reorder_list.txt` | `parts-clerk` (no `Write` tool) | **Missing** |
| `purchase_order.txt` | `order-writer` (has `Write`) | **Present** |

A file on disk is the strongest evidence you can get. It does not depend on what any agent claims in its answer.

**Step 2: attribute every write.** A file existing does not tell you *who* wrote it. So the cell walks through Cell 12's messages:

1. It builds a map from each Agent call's `id` to its `subagent_type`.
2. For every `ToolUseBlock` that can change files (`Write`, `Edit`, `Bash`), it looks at the message's `parent_tool_use_id`:
   - `None` means the **main agent** made the call.
   - Otherwise it is the `id` of the Agent call that started the subagent, so the map turns it into the agent's name.
3. It prints each call with its author and target path.

It also prints any `permission_denials` from the `ResultMessage`. These are tool calls that were refused during the run.

**How to read the outcome:**

- **`reorder_list.txt` missing and no `Write` from `parts-clerk`:** the restriction held. The `Write` tool was never in the clerk's session, so it could not even try.
- **`purchase_order.txt` written by `order-writer`:** the specialist that was given `Write` used it.
- **A write by the main agent:** the main agent did not respect the instruction. The cell says so plainly. This does not mean the clerk's restriction failed. It means a prompt instruction is a request, while a `tools` list is a hard limit. That difference is the lesson either way.

In [ ]:
from claude_agent_sdk import ResultMessage

# Step 1: filesystem ground truth
print("--- Files on disk ---")
for name in ("reorder_list.txt", "purchase_order.txt"):
    path = f"{ORDERS_DIR}/{name}"
    status = "PRESENT" if os.path.exists(path) else "missing"
    print(f"{name:<22}{status}")

# Step 2: attribute every file-changing tool call
agent_names = {call.id: call.input.get("subagent_type") for call in agent_calls(team_messages)}

print("\n--- Who made each file-changing call ---")
main_agent_wrote = False
found_any = False
for message in team_messages:
    if not isinstance(message, AssistantMessage):
        continue
    for block in message.content:
        if isinstance(block, ToolUseBlock) and block.name in ("Write", "Edit", "Bash"):
            found_any = True
            parent = message.parent_tool_use_id
            if parent is None:
                author = "main agent"
                main_agent_wrote = True
            else:
                author = agent_names.get(parent, f"unknown subagent ({parent[-6:]})")
            target = block.input.get("file_path") or block.input.get("command", "")
            print(f"{author:<16}{block.name:<7}{target}")

if not found_any:
    print("(no file-changing calls in the stream)")

if main_agent_wrote:
    print("\nThe MAIN AGENT made a file-changing call itself, despite the instruction.")
    print("The prompt asked it not to. A prompt is a request; a tools list is a hard limit.")

# Refused tool calls, if any
for message in team_messages:
    if isinstance(message, ResultMessage):
        print(f"\npermission_denials: {message.permission_denials or 'none'}")

--- Files on disk ---
reorder_list.txt      missing
purchase_order.txt    PRESENT

--- Who made each file-changing call ---
order-writer    Write  /content/bikecoop/orders/purchase_order.txt

permission_denials: none


## Cell 14: Define an agent in a markdown file

`AgentDefinition` is one way to define a specialist. The other is a **markdown file** in a `.claude/agents/` folder. A file-defined agent lives on disk, so it can be shared and reused without changing any Python code.

The file has two parts:

```
---
name: shift-planner
description: ...
tools: Read, Glob, Grep
model: claude-haiku-5-5
---

The agent's instructions go here, as plain text.
```

| Part | Meaning | `AgentDefinition` equivalent |
|---|---|---|
| `name` | The agent's name (required) | The key in the `agents` dictionary |
| `description` | When to use the agent (required) | `description` |
| `tools` | A comma-separated list of tools | `tools` |
| `model` | The model to use | `model` |
| The body below the header | The agent's instructions | `prompt` |

**Design decisions:**

- **Location.** The file goes in `/content/.claude/agents/`. The `/content` part matches the `cwd="/content"` the agents run with, because project agents are discovered relative to the working directory.
- **The model line is filled in from `MODEL_NAME`.** The file is written with a Python f-string, so changing Cell 3 changes this agent too.
- **Read-only tools.** The planner only needs to read work orders and the volunteer roster.

The cell writes the file and prints it, so you can see exactly what will be loaded.

In [ ]:
AGENT_DIR = "/content/.claude/agents"
os.makedirs(AGENT_DIR, exist_ok=True)

shift_planner_md = f"""---
name: shift-planner
description: Matches open work orders to volunteers by skill and shift day. Use when deciding which volunteer should take a work order.
tools: Read, Glob, Grep
model: {MODEL_NAME}
---

You are the shift planner at the Bike Repair Co-op.
Work orders are in /content/bikecoop/work_orders/ and the volunteer roster is
/content/bikecoop/volunteers.txt. Read the work order, then pick the volunteer
whose skills best fit the problem. Name the volunteer, their shift days, and
give one sentence explaining the match.
"""

with open(AGENT_FILE, "w", encoding="utf-8") as file:
    file.write(shift_planner_md)

print(f"--- {AGENT_FILE} ---")
with open(AGENT_FILE, encoding="utf-8") as file:
    print(file.read())

--- /content/.claude/agents/shift-planner.md ---
---
name: shift-planner
description: Matches open work orders to volunteers by skill and shift day. Use when deciding which volunteer should take a work order.
tools: Read, Glob, Grep
model: claude-haiku-5-5
---

You are the shift planner at the Bike Repair Co-op.
Work orders are in /content/bikecoop/work_orders/ and the volunteer roster is
/content/bikecoop/volunteers.txt. Read the work order, then pick the volunteer
whose skills best fit the problem. Name the volunteer, their shift days, and
give one sentence explaining the match.



## Cell 15: Load the file-defined agent and use it

This `query()` passes **no** `agents=` dictionary. Instead it sets `setting_sources=["project"]`, which tells the SDK to load project settings from the filesystem, including agent files in `.claude/agents/` under the working directory.

| Option | Value | Why |
|---|---|---|
| `setting_sources` | `["project"]` | Load project-level files from `/content/.claude/`. User and local settings are left out. |
| `allowed_tools` | `Read`, `Glob`, `Grep`, `Agent` | Read-only tools, plus `Agent` so delegation is auto-approved |
| `env` | `CLAUDE_CODE_DISABLE_BACKGROUND_TASKS` set to `"1"` | The main agent waits for the planner's answer |
| `cwd` | `"/content"` | The folder that contains `.claude/` |

**A detail worth knowing.** If you leave `setting_sources` out entirely, the SDK loads Claude Code's default sources, which are user, project and local. So an agent file in `/content/.claude/agents/` would also be visible to the earlier cells if it existed when they ran. That is why Cell 4 deletes it at the start. Setting `setting_sources=["project"]` makes the choice explicit and leaves out user and local settings.

**How to confirm the agent loaded.** The first message in the stream is a `SystemMessage` with subtype `"init"`. Its `data["agents"]` lists the agents available in the session. Entries may be plain strings or dictionaries with a `"name"` key, so the cell handles both, and prints the raw type of the entries so you can see which form your version uses.

The prompt names the agent explicitly:

```
Use the shift-planner agent to suggest which volunteer should take WO-103.
```

The cell prints the loaded agent names, confirms `shift-planner` is among them, then prints the `subagent_type` used and the final answer.

**Precedence:** if an agent defined in code with `agents=` has the same name as a file-defined agent, the one defined in code wins.

**What to look for:** `shift-planner` in the loaded list, `subagent_type: shift-planner`, and a volunteer whose skills fit WO-103's problem (compare with the files printed in Cell 5). If `shift-planner` is missing from the list, the file did not load. Check the header block in Cell 14 first.

In [ ]:
from claude_agent_sdk import SystemMessage

file_options = ClaudeAgentOptions(
    setting_sources=["project"],
    allowed_tools=["Read", "Glob", "Grep", "Agent"],
    env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},
    model=MODEL_NAME,
    cwd="/content",
)

file_messages = []
loaded_agents = []
try:
    async for message in query(
        prompt="Use the shift-planner agent to suggest which volunteer should take WO-103.",
        options=file_options,
    ):
        file_messages.append(message)
        if isinstance(message, SystemMessage) and message.subtype == "init":
            raw_agents = message.data.get("agents", [])
            if raw_agents:
                print(f"Entry type in data['agents']: {type(raw_agents[0]).__name__}")
            for entry in raw_agents:
                if isinstance(entry, str):
                    loaded_agents.append(entry)
                elif isinstance(entry, dict):
                    loaded_agents.append(entry.get("name", ""))
except Exception as error:
    print(f"Run ended with an error: {error}")

print(f"Agents loaded: {loaded_agents}")
if "shift-planner" in loaded_agents:
    print("shift-planner was loaded from the markdown file.")
else:
    print("shift-planner was NOT loaded. Check the file written in Cell 14.")

print("\n--- Delegations ---")
calls = agent_calls(file_messages)
if calls:
    for call in calls:
        print(f"subagent_type: {call.input.get('subagent_type')}")
else:
    print("no delegation")

print("\n--- Final answer ---")
print(final_text(file_messages) or "(no answer returned)")

Entry type in data['agents']: str
Agents loaded: ['claude', 'Explore', 'general-purpose', 'Plan', 'shift-planner', 'statusline-setup']
shift-planner was loaded from the markdown file.

--- Delegations ---
subagent_type: shift-planner

--- Final answer ---
The shift-planner recommends **Asha** for WO-103.

- **Skill match:** WO-103 is a flat tyre that needs a new tube, which falls under wheels. Asha's listed skill is wheels and brakes, the closest match on the roster.
- **Shift fit:** Asha works Mondays and Thursdays. Today is Wednesday 18 November 2026, so her next shift is Thursday 19 November.

**Alternates:**
- **Esha** is the currently assigned volunteer. Her skill is general servicing, which can cover a tube swap, and she is on shift today (Monday and Wednesday). Leaving the job with her avoids a handover, but she is a generalist rather than a wheels specialist.
- **Farid** (Thursday and Sunday) is a weaker fit, since his skills are kids' bikes and fitting.

**Before scheduling:**